# **Predicting Park Access Rankings Across U.S. Cities**
**Authors**: Jingyuan Liu, Shivani Aggarwal, Sarenna Ulman, Luna Gulec

## Summary

## Introduction

#### Background Information

#### Research Question

#### About the Dataset

#### Data Dictionary
**Note that "points" are essentially their yearly normalized values (higher points = better).**

|variable                  |class     |description |
|:-------------------------|:---------|:-----------|
|year                      |double    | Year of measurement |
|rank                      |double    | Yearly rank |
|city                      |character | City Name |
|med_park_size_data        |double    | Median park size acres |
|med_park_size_points      |double    | Median park size in points |
|park_pct_city_data        |character | Parkland as percentage of city area |
|park_pct_city_points      |double    | Parkland as % of city area points |
|pct_near_park_data        |character | Percent of residents within a 10 minute walk to park |
|pct_near_park_points      |double    | Percent of residents within a 10 minute walk to park points |
|spend_per_resident_data   |character | Spending per resident in USD |
|spend_per_resident_points |double    | Spending per resident in points |
|basketball_data           |double    | Basketball hoops per 10,000 residents |
|basketball_points         |double    | Basketball hoops per 10,000 residents points |
|dogpark_data              |double    | Dog parks per 100,000 residents|
|dogpark_points            |double    | Dog parks per 100,000 residents points |
|playground_data           |double    | Playgrounds per 10,000 residents |
|playground_points         |double    | Playgrounds per 10,000 residents points |
|rec_sr_data               |double    | Recreation and senior centers per 20,000 residents |
|rec_sr_points             |double    | Recreation and senior centers per 20,000 residents points |
|restroom_data             |double    | Restrooms per 10,000 residents |
|restroom_points           |double    | Restrooms per 10,000 residents points |
|splashground_data         |double    | Splashgrounds and splashpads per 100,000 residents |
|splashground_points       |double    | Splashgrounds and splashpads per 100,000 residents points |
|amenities_points          |double    | Amenities points total (ie play areas) |
|total_points              |double    | Total points (varies in denominator per/year) |
|total_pct                 |double    | Total points as a percentage|
|city_dup                  |character | City duplicated name |
|park_benches              |double    | Number of park benches|
This can also be found at the data website [here](https://github.com/rfordatascience/tidytuesday/blob/main/data/2021/2021-06-22/readme.md).

## Methods & Results

First, we need to import all necessary packages for this analysis.

In [14]:
import pandas as pd

#### Data Wrangling and Cleaning

We then load data from the original source on the web. Here, we present the first 5 rows of the raw data.

In [15]:
url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2021/2021-06-22/parks.csv"
data_raw = pd.read_csv(url)
data_raw.head()

,year,rank,city,med_park_size_data,med_park_size_points,park_pct_city_data,park_pct_city_points,pct_near_park_data,pct_near_park_points,spend_per_resident_data,...,rec_sr_points,restroom_data,restroom_points,splashground_data,splashground_points,amenities_points,total_points,total_pct,city_dup,park_benches
0,2020,1,Minneapolis,5.7,26.0,15%,38.0,98%,98.0,$319,...,100.0,2.9,91.0,4.0,100.0,79.3,341.0,85.3,Minneapolis,NaN
1,2020,2,"Washington, D.C.",1.4,5.0,24%,50.0,98%,98.0,$307,...,100.0,2.6,80.0,4.1,100.0,80.3,333.0,83.3,"Washington, D.C.",NaN
2,2020,3,St. Paul,3.2,14.0,15%,39.0,99%,99.0,$219,...,100.0,3.2,100.0,1.3,47.0,78.0,330.0,82.5,St. Paul,NaN
3,2020,4,"Arlington, Virginia",2.4,10.0,11%,28.0,99%,99.0,$301,...,76.0,3.1,97.0,2.2,81.0,89.0,326.0,81.5,"Arlington, Virginia",NaN
4,2020,5,Cincinnati,4.4,20.0,14%,36.0,82%,74.0,$190,...,100.0,3.2,98.0,4.5,100.0,92.5,323.0,80.6,Cincinnati,NaN


In [16]:
data_raw.shape

(713, 28)

We can see that the original raw dataset has 713 rows and 28 columns. Then, we will perform data wrangling to clean the data from it’s original format to the format necessary for the purpose of this analysis.

Since this dataset records the yearly rankings of park access across U.S. cities, it is inherently time-series data. However, advanced time-series methods are beyond the scope of this course. Instead, we will extract each city's rank from the last time, denoted as `rank_last_time`, to capture basic year-to-year trends. For the first observation year of a city, we decide to impute its `rank_last_time` with its ranking in the current year (i.e., `rank`).

In [17]:
data_processed = data_raw.sort_values(["city", "year"])
data_processed["rank_last_time"] = data_processed.groupby("city")["rank"].shift(1)
data_processed['rank_last_time'] = data_processed['rank_last_time'].fillna(data_processed['rank'])

We will then remove data from year 2012, 2013, and 2014 because there are so many missing values in those rows, which will not help with building a predictive model.

In [18]:
data_processed = data_processed[~data_processed["year"].isin([2012,2013,2014])]

We noticed that more than 10% of the data in columns `restroom_data`, `restroom_points`, `splashground_data`, `splashground_points`, `total_points`, `total_pct`, `city_dup`, and `park_benches` are missing, with some columns approaching 50%. Therefore, we decided to drop these columns as advanced data imputation methods are also beyond the scope of this course.

In [19]:
data_processed.isna().sum()
data_processed = data_processed.drop(columns=['restroom_data','restroom_points','splashground_data',
                                              'splashground_points','total_points','total_pct',
                                              'city_dup','park_benches'])

There are a total of more than 100 cities in this dataset, but each city only has fewer than 10 observations. This makes `city` not an ideal categorical variable for regression due to the potential overfitting issue. Therefore, we perform feature aggregation and add a new column called `state` so that each category has more valid observations. We will manually map each city into a state.

In [20]:
data_processed['city'].nunique()

101

In [21]:
city_to_state = {
    'Albuquerque': 'NM', 'Anaheim': 'CA', 'Anchorage': 'AK', 
    'Arlington, Texas': 'TX', 'Arlington, Virginia': 'VA', 'Atlanta': 'GA', 
    'Aurora': 'CO', 'Austin': 'TX', 'Bakersfield': 'CA', 'Baltimore': 'MD', 
    'Baton Rouge': 'LA', 'Boise': 'ID', 'Boston': 'MA', 'Buffalo': 'NY', 
    'Chandler': 'AZ', 'Charlotte': 'NC', 'Charlotte/Mecklenburg County': 'NC',
    'Chesapeake': 'VA', 'Chicago': 'IL', 'Chula Vista': 'CA', 'Cincinnati': 'OH', 
    'Cleveland': 'OH', 'Colorado Springs': 'CO', 'Columbus': 'OH', 
    'Corpus Christi': 'TX', 'Dallas': 'TX', 'Denver': 'CO', 'Des Moines': 'IA', 
    'Detroit': 'MI', 'Durham': 'NC', 'El Paso': 'TX', 'Fort Wayne': 'IN', 
    'Fort Worth': 'TX', 'Fremont': 'CA', 'Fresno': 'CA', 'Garland': 'TX', 
    'Glendale': 'AZ', 'Greensboro': 'NC', 'Henderson': 'NV', 'Hialeah': 'FL', 
    'Honolulu': 'HI', 'Houston': 'TX', 'Indianapolis': 'IN', 'Irvine': 'CA', 
    'Irving': 'TX', 'Jacksonville': 'FL', 'Jersey City': 'NJ', 'Kansas City': 'MO', 
    'Laredo': 'TX', 'Las Vegas': 'NV', 'Lexington': 'KY', 'Lincoln': 'NE', 
    'Long Beach': 'CA', 'Los Angeles': 'CA', 'Louisville': 'KY', 'Lubbock': 'TX', 
    'Madison': 'WI', 'Memphis': 'TN', 'Mesa': 'AZ', 'Miami': 'FL', 
    'Milwaukee': 'WI', 'Minneapolis': 'MN', 'Nashville': 'TN', 'New Orleans': 'LA', 
    'New York': 'NY', 'Newark': 'NJ', 'Norfolk': 'VA', 'North Las Vegas': 'NV', 
    'Oakland': 'CA', 'Oklahoma City': 'OK', 'Omaha': 'NE', 'Orlando': 'FL', 
    'Philadelphia': 'PA', 'Phoenix': 'AZ', 'Pittsburgh': 'PA', 'Plano': 'TX', 
    'Portland': 'OR', 'Raleigh': 'NC', 'Reno': 'NV', 'Richmond': 'VA', 
    'Riverside': 'CA', 'Sacramento': 'CA', 'San Antonio': 'TX', 'San Diego': 'CA', 
    'San Francisco': 'CA', 'San Jose': 'CA', 'Santa Ana': 'CA', 'Scottsdale': 'AZ', 
    'Seattle': 'WA', 'St. Louis': 'MO', 'St. Paul': 'MN', 'St. Petersburg': 'FL', 
    'Stockton': 'CA', 'Tampa': 'FL', 'Toledo': 'OH', 'Tucson': 'AZ', 'Tulsa': 'OK', 
    'Virginia Beach': 'VA', 'Washington, D.C.': 'DC', 'Wichita': 'KS', 
    'Winston-Salem': 'NC'
}
data_processed['state'] = data_processed['city'].map(city_to_state)

Finally, we will convert columns `year`, `city`, and `state` to type `category`, and columns `rank` and `rank_last_time` to type `int`. In addition, we know that variables ending with "points" are essentially their yearly normalized values (higher points = better), so we decided to remove raw numerical variables (i.e., variables ending with `"_data"`) and just keep the normalized ones with potential categorical variables in the model. In particular, our response variable is **`rank`**!

In [22]:
data_processed['year'] = data_processed['year'].astype('category')
data_processed['city'] = data_processed['city'].astype('category')
data_processed['state'] = data_processed['state'].astype('category')
data_processed['rank'] = data_processed['rank'].astype('int')
data_processed['rank_last_time'] = data_processed['rank_last_time'].astype('int')
data_processed = data_processed.drop(columns=data_processed.filter(regex='data$').columns)

Finally, we can review the processed dataset and export it as a CSV to the `/data/processed` directory, and now we are ready to proceed!

In [23]:
data_processed.head(10)

,year,rank,city,med_park_size_points,park_pct_city_points,pct_near_park_points,spend_per_resident_points,basketball_points,dogpark_points,playground_points,rec_sr_points,amenities_points,rank_last_time,state
500,2015,13,Albuquerque,8.0,20.0,32.0,7.0,8.0,20.0,10.0,15.0,53.0,15,NM
409,2016,20,Albuquerque,8.0,20.0,31.0,6.0,8.0,20.0,11.0,14.0,13.0,13,NM
307,2017,17,Albuquerque,8.0,20.0,31.0,6.0,7.0,20.0,11.0,13.0,13.0,20,NM
233,2018,40,Albuquerque,8.0,20.0,30.0,10.0,11.0,40.0,24.0,27.0,21.0,17,NM
130,2019,34,Albuquerque,20.0,50.0,82.5,27.5,25.0,100.0,57.5,68.0,52.1,40,NM
34,2020,35,Albuquerque,19.0,50.0,81.0,30.0,27.0,100.0,52.0,91.0,60.8,34,NM
538,2015,51,Anaheim,17.0,7.0,22.0,6.0,4.0,3.0,3.0,2.0,12.0,36,CA
454,2016,65,Anaheim,17.0,8.0,19.0,7.0,4.0,2.0,4.0,2.0,3.0,51,CA
353,2017,63,Anaheim,17.0,8.0,19.0,7.0,3.0,4.0,4.0,2.0,3.0,65,CA
261,2018,68,Anaheim,16.0,8.0,19.0,13.0,4.0,7.0,7.0,5.0,11.0,63,CA


In [24]:
data_processed.to_csv("../data/processed/parks_processed.csv", index=False)

#### Exploratory Data Analysis & Visualization

#### Regression Analysis 

#### Results & Visualizations

## Discussion

## References